# Grouped-Query Attention (GQA)

While MQA significantly reduces memory usage, it can sometimes degrade model quality. DeepSeek models use Grouped-Query Attention (GQA) as a balanced approach:

- Group attention heads into clusters (e.g., 4-8 groups for 32 heads)
- Each group shares the same K,V projections
- Queries remain separate for each head

GQA offers a favorable trade-off:
- Better quality than MQA (more expressive)
- More efficient than standard multi-head attention
- Well-suited for DeepSeek's massive scale

This approach is particularly important for DeepSeek's 671B parameter models where balancing efficiency and quality is crucial.

## GQA — Grouped-Query Attention

**Grouped-Query Attention (GQA)** is a compromise between **MHA** and **MQA**.

Instead of:

* **MHA:** Every Q head has its own K/V
* **MQA:** All Q heads share **one** K/V

GQA **groups multiple Q heads together**, and each group shares one K/V pair.


**MHA**:

```text
Q₁ → K₁,V₁
Q₂ → K₂,V₂
Q₃ → K₃,V₃
Q₄ → K₄,V₄
```

**MQA**:

```text
Q₁  ─┐
Q₂  ─┤
Q₃  ─┤──> Shared K, V
Q₄  ─┘
```

**GQA**:

```text
Q₁ ─┐
Q₂ ─┘ → K₁,V₁

Q₃ ─┐
Q₄ ─┘ → K₂,V₂
```

### So if you have **8 Q heads and 2 KV heads**:

> 4 Groups = 8 Q-heads / 2 KV-heads

```text
8 Query heads
      ↓
2 Key/Value heads

Q₁, Q₂ → K₁, V₁
Q₃, Q₄ → K₂, V₂
Q₅, Q₆ → K₃, V₃
Q₇, Q₈ → K₄, V₄
```

### Why GQA?

It provides a **middle ground**:

> **Much smaller KV cache than MHA, while retaining more model quality than MQA.**

**MHA → Maximum quality, maximum KV memory**

**MQA → Minimum KV memory, potentially lower quality**

**GQA → Balance between the two (Quality & Memory)** ⚖️


## Listing 2.4: Implementing Grouped-Query Attention (GQA)

The code below implements a Grouped-Query Attention layer from scratch. Note the key differences:

1. Query projection (`self.W_q`) still maps to the full model dimension
2. Key and value projections (`self.W_k` and `self.W_v`) map to `self.num_groups * self.d_head`
3. We use `repeat_interleave()` to match each key and value group with its corresponding query heads

This implementation demonstrates how GQA balances between standard multi-head attention and MQA:
- Fewer K,V projections than standard attention (reduced by factor of `num_heads/num_groups`)
- More K,V diversity than MQA (one set per group rather than just one shared)

DeepSeek models use this approach to maintain quality while reducing memory requirements, especially for the KV cache.

In [1]:
import torch
import torch.nn as nn

# ===================================================
# LISTING 2.4: Implementing a GQA layer from scratch
# ===================================================
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_groups, dropout=0.0, max_seq_len: int = 1024):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        assert num_heads % num_groups == 0, "num_heads must be divisible by num_groups"

        self.d_model = d_model
        self.num_heads = num_heads
        self.num_groups = num_groups
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, self.num_groups * self.d_head) # Grouped projection for K
        self.W_v = nn.Linear(d_model, self.num_groups * self.d_head) # Grouped projection for V
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self._register_mask_buffer(max_seq_len)

    def _register_mask_buffer(self, max_seq_len):
        if max_seq_len > 0:
            mask = torch.triu(torch.ones(1, 1, max_seq_len, max_seq_len, dtype=torch.bool), diagonal=1)
            self.register_buffer("causal_mask", mask, persistent=False)
        else:
            self.causal_mask = None

    def _get_causal_mask(self, seq_len, device):
        if self.causal_mask is not None and self.causal_mask.size(-1) >= seq_len:
            return self.causal_mask[:, :, :seq_len, :seq_len]
        # Dynamically create mask if needed
        return torch.triu(torch.ones(1, 1, seq_len, seq_len, dtype=torch.bool, device=device), diagonal=1)

    def forward(self, x):
        B, T, _ = x.shape

        # Query: (B, num_heads, T, d_head)
        q = self.W_q(x).view(B, T, self.num_heads, self.d_head).transpose(1, 2)

        # Key & Value: (B, num_groups, T, d_head)
        k = self.W_k(x).view(B, T, self.num_groups, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.num_groups, self.d_head).transpose(1, 2)

        heads_per_group = self.num_heads // self.num_groups

        # Repeat K and V to match query heads
        k = k.repeat_interleave(heads_per_group, dim=1) # (B, num_heads, T, d_head)
        v = v.repeat_interleave(heads_per_group, dim=1) # (B, num_heads, T, d_head)

        attn_scores = (q @ k.transpose(-2, -1)) * (self.d_head**-0.5)

        causal_mask = self._get_causal_mask(T, x.device)
        attn_scores = attn_scores.masked_fill(causal_mask, float("-inf"))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context = (attn_weights @ v).transpose(1, 2).contiguous().view(B, T, self.d_model)

        return self.W_o(context)


# -----------------------------------------------
# ---------------- Usage Example ----------------
# -----------------------------------------------
d_model = 512
num_heads = 32
num_groups = 4   # e.g., Llama 2 7B uses 4 groups for 32 heads
batch_size = 4
seq_len = 64

gqa_layer = GroupedQueryAttention(d_model, num_heads, num_groups)
dummy_input = torch.randn(batch_size, seq_len, d_model)
output = gqa_layer(dummy_input)


print("✅ GQA Layer successful! 🎉")
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

✅ GQA Layer successful! 🎉
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])


### The Dark Side of GQA

The main downside of **GQA** is that **multiple Query heads must share the same K/V representations within each group**.

> **It reduces KV-cache memory and speeds up inference, but may slightly reduce model quality compared with full MHA.**

So the trade-off is:

**🚀 Faster + 💾 Less memory → potentially 📉 Slightly less expressive attention**

### Comparison of Attention Variants

| Feature | Standard Multi-Head | Grouped-Query | Multi-Query |
|---------|---------------------|---------------|-------------|
| K,V Projections | One per head | One per group | One shared |
| Cache Size | Largest | Medium | Smallest |
| Quality | Highest | Good | Lower |
| Memory Efficiency | Low | Medium | High |

DeepSeek models use these optimizations strategically depending on the model size and intended use case.

## 2.5 Conclusion: The Key-Value Cache as Foundation

The key-value cache represents the first major breakthrough in addressing the inference bottleneck for transformer models, and it serves as the foundation for all subsequent attention optimizations in DeepSeek models:

1. **Fundamental Optimization**: By storing and reusing key-value pairs, the KV cache dramatically reduces the computational cost of autoregressive generation.

2. **Memory-Computation Tradeoff**: The KV cache exemplifies an essential engineering principle—trading increased memory usage for reduced computation, which is often beneficial for practical applications.

3. **Foundation for Advanced Techniques**: The MQA and GQA techniques build directly on the KV cache foundation, further optimizing memory usage while maintaining model quality.

4. **Enabling Long Context**: DeepSeek's impressive 128K token context window capability would be impossible without these attention optimizations.

Understanding the key-value cache and its evolved forms is essential for grasping how DeepSeek models achieve their remarkable balance of quality and efficiency at scale. As we'll see in the next chapter, these attention optimizations work in concert with DeepSeek's Mixture of Experts (MoE) architecture to enable its massive 671B parameter scale while keeping activated parameters at just 37B.